In [20]:
import pandas as pd
import joblib
import os

# Load the Model and the Scaler

model_path = os.path.join("..", "models", "diabetes_risk_model_v1.pkl")
scaler_path = os.path.join("..", "models", "feature_scaler_v1.pkl")

model = joblib.load(model_path)
scaler = joblib.load(scaler_path)

print("✅ Model and Scaler loaded successfully.")

✅ Model and Scaler loaded successfully.


In [21]:
# Feature order must match training (02_preprocessing_fe / X_train.csv)
FEATURE_ORDER = [
    'age', 'blood_pressure', 'fasting_glucose_level', 'insulin_level', 'cholesterol_level',
    'triglycerides_level', 'physical_activity_level', 'daily_calorie_intake', 'sugar_intake_grams_per_day',
    'sleep_hours', 'stress_level', 'family_history_diabetes', 'waist_circumference_cm', 'gender_Male',
    'health_habits_score', 'lipid_ratio', 'high_lipid_risk'
]

features_to_scale = [
    'age', 'blood_pressure', 'fasting_glucose_level', 'insulin_level', 'cholesterol_level',
    'triglycerides_level', 'daily_calorie_intake', 'sugar_intake_grams_per_day',
    'sleep_hours', 'stress_level', 'waist_circumference_cm', 'health_habits_score', 'lipid_ratio'
]

# Encodings (same as 02_preprocessing_fe.ipynb)
ACTIVITY_MAP = {'Low': 0, 'Moderate': 1, 'High': 2}
FAMILY_HISTORY_MAP = {'No': 0, 'Yes': 1}

In [22]:
# Get Risk Prediction function
def get_risk_prediction(patient_dict):
    # Encode categoricals
    physical_activity = patient_dict.get('physical_activity_level', 'Moderate')
    if isinstance(physical_activity, str):
        physical_activity = ACTIVITY_MAP.get(physical_activity, 1)
    family_history = patient_dict.get('family_history_diabetes', 'No')
    if isinstance(family_history, str):
        family_history = FAMILY_HISTORY_MAP.get(family_history, 0)
    gender = patient_dict.get('gender', 'Male')
    gender_Male = 1 if (gender == 'Male' or gender is True) else 0
    # Derived: high_lipid_risk, lipid_ratio if missing
    trig = patient_dict['triglycerides_level']
    chol = patient_dict['cholesterol_level']
    high_lipid_risk = 1 if (trig > 150 and chol > 200) else 0
    lipid_ratio = patient_dict.get('lipid_ratio', trig / chol if chol else 0)
    # Build row in training order
    row = {
        'age': patient_dict['age'],
        'blood_pressure': patient_dict['blood_pressure'],
        'fasting_glucose_level': patient_dict['fasting_glucose_level'],
        'insulin_level': patient_dict['insulin_level'],
        'cholesterol_level': chol,
        'triglycerides_level': trig,
        'physical_activity_level': physical_activity,
        'daily_calorie_intake': patient_dict['daily_calorie_intake'],
        'sugar_intake_grams_per_day': patient_dict['sugar_intake_grams_per_day'],
        'sleep_hours': patient_dict['sleep_hours'],
        'stress_level': patient_dict['stress_level'],
        'family_history_diabetes': family_history,
        'waist_circumference_cm': patient_dict['waist_circumference_cm'],
        'gender_Male': gender_Male,
        'health_habits_score': patient_dict['health_habits_score'],
        'lipid_ratio': lipid_ratio,
        'high_lipid_risk': high_lipid_risk,
    }
    df = pd.DataFrame([row])
    df = df[FEATURE_ORDER]
    df[features_to_scale] = scaler.transform(df[features_to_scale])
    pred = model.predict(df)[0]
    prob = model.predict_proba(df)[0]
    risk_labels = {0: "Low Risk", 1: "Prediabetes", 2: "High Risk"}
    return risk_labels[pred], prob



In [23]:
# Testing
sample_patient = {
    'age': 30, 'blood_pressure': 100, 'fasting_glucose_level': 80,
    'insulin_level': 22.0, 'cholesterol_level': 200, 'triglycerides_level': 190,
    'physical_activity_level': 'Low', 'daily_calorie_intake': 3000, 'sugar_intake_grams_per_day': 100,
    'sleep_hours': 8, 'stress_level': 5, 'family_history_diabetes': 'No', 'waist_circumference_cm': 80,
    'gender': 'Male', 'health_habits_score': 30, 'lipid_ratio': 190/240
}

result, confidence = get_risk_prediction(sample_patient)
print(f"Result: {result} | Confidence: {max(confidence)*100:.2f}%")

Result: Prediabetes | Confidence: 100.00%


In [ ]:
# Defining a list of 20 diverse patient profiles
stress_test_patients = [
    # --- PROBABLY LOW RISK ---
    {'age': 25, 'blood_pressure': 50, 'fasting_glucose_level': 80, 'insulin_level': 10, 'cholesterol_level': 160, 'triglycerides_level': 100, 'physical_activity_level': 'High', 'daily_calorie_intake': 2100, 'sugar_intake_grams_per_day': 25, 'sleep_hours': 8, 'stress_level': 2, 'family_history_diabetes': 'No', 'waist_circumference_cm': 75, 'gender': 'Female', 'health_habits_score': 90},
    {'age': 30, 'blood_pressure': 115, 'fasting_glucose_level': 85, 'insulin_level': 12, 'cholesterol_level': 175, 'triglycerides_level': 110, 'physical_activity_level': 'Moderate', 'daily_calorie_intake': 2000, 'sugar_intake_grams_per_day': 30, 'sleep_hours': 7, 'stress_level': 3, 'family_history_diabetes': 'No', 'waist_circumference_cm': 80, 'gender': 'Male', 'health_habits_score': 80},
    {'age': 22, 'blood_pressure': 105, 'fasting_glucose_level': 78, 'insulin_level': 8, 'cholesterol_level': 150, 'triglycerides_level': 90, 'physical_activity_level': 'High', 'daily_calorie_intake': 2300, 'sugar_intake_grams_per_day': 20, 'sleep_hours': 9, 'stress_level': 1, 'family_history_diabetes': 'No', 'waist_circumference_cm': 70, 'gender': 'Female', 'health_habits_score': 95},
    {'age': 40, 'blood_pressure': 120, 'fasting_glucose_level': 90, 'insulin_level': 15, 'cholesterol_level': 190, 'triglycerides_level': 130, 'physical_activity_level': 'Moderate', 'daily_calorie_intake': 1800, 'sugar_intake_grams_per_day': 35, 'sleep_hours': 7, 'stress_level': 4, 'family_history_diabetes': 'No', 'waist_circumference_cm': 85, 'gender': 'Female', 'health_habits_score': 75},
    {'age': 35, 'blood_pressure': 118, 'fasting_glucose_level': 88, 'insulin_level': 14, 'cholesterol_level': 180, 'triglycerides_level': 120, 'physical_activity_level': 'High', 'daily_calorie_intake': 2500, 'sugar_intake_grams_per_day': 40, 'sleep_hours': 8, 'stress_level': 2, 'family_history_diabetes': 'No', 'waist_circumference_cm': 82, 'gender': 'Male', 'health_habits_score': 85},

    # --- PROBABLY PREDIABETES (The "Fuzzy" zone) ---
    {'age': 45, 'blood_pressure': 135, 'fasting_glucose_level': 110, 'insulin_level': 18, 'cholesterol_level': 210, 'triglycerides_level': 160, 'physical_activity_level': 'Low', 'daily_calorie_intake': 2600, 'sugar_intake_grams_per_day': 70, 'sleep_hours': 6, 'stress_level': 6, 'family_history_diabetes': 'Yes', 'waist_circumference_cm': 95, 'gender': 'Male', 'health_habits_score': 50},
    {'age': 50, 'blood_pressure': 130, 'fasting_glucose_level': 105, 'insulin_level': 20, 'cholesterol_level': 220, 'triglycerides_level': 170, 'physical_activity_level': 'Moderate', 'daily_calorie_intake': 2400, 'sugar_intake_grams_per_day': 60, 'sleep_hours': 5, 'stress_level': 7, 'family_history_diabetes': 'No', 'waist_circumference_cm': 92, 'gender': 'Female', 'health_habits_score': 45},
    {'age': 38, 'blood_pressure': 128, 'fasting_glucose_level': 102, 'insulin_level': 17, 'cholesterol_level': 205, 'triglycerides_level': 155, 'physical_activity_level': 'Low', 'daily_calorie_intake': 2800, 'sugar_intake_grams_per_day': 85, 'sleep_hours': 6, 'stress_level': 8, 'family_history_diabetes': 'Yes', 'waist_circumference_cm': 98, 'gender': 'Male', 'health_habits_score': 40},
    {'age': 55, 'blood_pressure': 140, 'fasting_glucose_level': 115, 'insulin_level': 22, 'cholesterol_level': 230, 'triglycerides_level': 180, 'physical_activity_level': 'Low', 'daily_calorie_intake': 2500, 'sugar_intake_grams_per_day': 65, 'sleep_hours': 5, 'stress_level': 6, 'family_history_diabetes': 'No', 'waist_circumference_cm': 102, 'gender': 'Female', 'health_habits_score': 55},
    {'age': 42, 'blood_pressure': 132, 'fasting_glucose_level': 108, 'insulin_level': 19, 'cholesterol_level': 215, 'triglycerides_level': 165, 'physical_activity_level': 'Moderate', 'daily_calorie_intake': 2200, 'sugar_intake_grams_per_day': 55, 'sleep_hours': 7, 'stress_level': 5, 'family_history_diabetes': 'Yes', 'waist_circumference_cm': 94, 'gender': 'Male', 'health_habits_score': 60},

    # --- PROBABLY HIGH RISK ---
    {'age': 65, 'blood_pressure': 160, 'fasting_glucose_level': 150, 'insulin_level': 35, 'cholesterol_level': 280, 'triglycerides_level': 250, 'physical_activity_level': 'Low', 'daily_calorie_intake': 3500, 'sugar_intake_grams_per_day': 130, 'sleep_hours': 4, 'stress_level': 9, 'family_history_diabetes': 'Yes', 'waist_circumference_cm': 118, 'gender': 'Male', 'health_habits_score': 15},
    {'age': 70, 'blood_pressure': 150, 'fasting_glucose_level': 160, 'insulin_level': 40, 'cholesterol_level': 260, 'triglycerides_level': 240, 'physical_activity_level': 'Low', 'daily_calorie_intake': 3000, 'sugar_intake_grams_per_day': 100, 'sleep_hours': 5, 'stress_level': 10, 'family_history_diabetes': 'Yes', 'waist_circumference_cm': 110, 'gender': 'Female', 'health_habits_score': 20},
    {'age': 58, 'blood_pressure': 155, 'fasting_glucose_level': 140, 'insulin_level': 30, 'cholesterol_level': 250, 'triglycerides_level': 230, 'physical_activity_level': 'Low', 'daily_calorie_intake': 3200, 'sugar_intake_grams_per_day': 150, 'sleep_hours': 4, 'stress_level': 8, 'family_history_diabetes': 'Yes', 'waist_circumference_cm': 112, 'gender': 'Male', 'health_habits_score': 10},
    {'age': 60, 'blood_pressure': 145, 'fasting_glucose_level': 135, 'insulin_level': 25, 'cholesterol_level': 240, 'triglycerides_level': 220, 'physical_activity_level': 'Moderate', 'daily_calorie_intake': 2900, 'sugar_intake_grams_per_day': 90, 'sleep_hours': 6, 'stress_level': 7, 'family_history_diabetes': 'Yes', 'waist_circumference_cm': 108, 'gender': 'Female', 'health_habits_score': 30},
    {'age': 52, 'blood_pressure': 148, 'fasting_glucose_level': 142, 'insulin_level': 28, 'cholesterol_level': 270, 'triglycerides_level': 235, 'physical_activity_level': 'Low', 'daily_calorie_intake': 3100, 'sugar_intake_grams_per_day': 110, 'sleep_hours': 5, 'stress_level': 9, 'family_history_diabetes': 'Yes', 'waist_circumference_cm': 114, 'gender': 'Male', 'health_habits_score': 25},

    # --- OUTLIERS / EDGE CASES ---
    {'age': 18, 'blood_pressure': 120, 'fasting_glucose_level': 130, 'insulin_level': 20, 'cholesterol_level': 180, 'triglycerides_level': 140, 'physical_activity_level': 'High', 'daily_calorie_intake': 2000, 'sugar_intake_grams_per_day': 40, 'sleep_hours': 8, 'stress_level': 3, 'family_history_diabetes': 'Yes', 'waist_circumference_cm': 80, 'gender': 'Male', 'health_habits_score': 80}, # Young but high glucose
    {'age': 80, 'blood_pressure': 110, 'fasting_glucose_level': 90, 'insulin_level': 12, 'cholesterol_level': 190, 'triglycerides_level': 120, 'physical_activity_level': 'Moderate', 'daily_calorie_intake': 1600, 'sugar_intake_grams_per_day': 20, 'sleep_hours': 7, 'stress_level': 2, 'family_history_diabetes': 'No', 'waist_circumference_cm': 85, 'gender': 'Female', 'health_habits_score': 70}, # Old but healthy stats
    {'age': 40, 'blood_pressure': 140, 'fasting_glucose_level': 100, 'insulin_level': 15, 'cholesterol_level': 210, 'triglycerides_level': 200, 'physical_activity_level': 'Low', 'daily_calorie_intake': 3000, 'sugar_intake_grams_per_day': 100, 'sleep_hours': 6, 'stress_level': 8, 'family_history_diabetes': 'No', 'waist_circumference_cm': 100, 'gender': 'Male', 'health_habits_score': 40}, # High lipids/lifestyle, normal glucose
    {'age': 35, 'blood_pressure': 115, 'fasting_glucose_level': 120, 'insulin_level': 25, 'cholesterol_level': 180, 'triglycerides_level': 110, 'physical_activity_level': 'High', 'daily_calorie_intake': 2200, 'sugar_intake_grams_per_day': 50, 'sleep_hours': 8, 'stress_level': 5, 'family_history_diabetes': 'Yes', 'waist_circumference_cm': 88, 'gender': 'Female', 'health_habits_score': 65}, # Fit but high glucose (Genetic?)
    {'age': 45, 'blood_pressure': 130, 'fasting_glucose_level': 95, 'insulin_level': 14, 'cholesterol_level': 200, 'triglycerides_level': 150, 'physical_activity_level': 'Moderate', 'daily_calorie_intake': 2100, 'sugar_intake_grams_per_day': 45, 'sleep_hours': 7, 'stress_level': 4, 'family_history_diabetes': 'No', 'waist_circumference_cm': 90, 'gender': 'Male', 'health_habits_score': 70}, # Borderline everything
]

# Run the loop
results = []
for i, patient in enumerate(stress_test_patients):
    res, prob = get_risk_prediction(patient)
    results.append({'ID': i+1, 'Prediction': res, 'Confidence': f"{max(prob)*100:.2f}%"})

# View results as a table
results_df = pd.DataFrame(results)
print(results_df)

    ID   Prediction Confidence
0    1  Prediabetes    100.00%
1    2  Prediabetes    100.00%
2    3  Prediabetes    100.00%
3    4  Prediabetes    100.00%
4    5  Prediabetes    100.00%
5    6  Prediabetes    100.00%
6    7  Prediabetes    100.00%
7    8  Prediabetes    100.00%
8    9  Prediabetes    100.00%
9   10  Prediabetes    100.00%
10  11  Prediabetes    100.00%
11  12  Prediabetes    100.00%
12  13  Prediabetes    100.00%
13  14  Prediabetes    100.00%
14  15  Prediabetes    100.00%
15  16  Prediabetes    100.00%
16  17  Prediabetes    100.00%
17  18  Prediabetes    100.00%
18  19  Prediabetes    100.00%
19  20  Prediabetes    100.00%


In [25]:
import numpy as np

# Load processed data and model
X_test = pd.read_csv("../data/processed/X_test.csv")
model = joblib.load("../models/diabetes_risk_model_v1.pkl")

# Take 50 random samples
sample_50 = X_test.sample(n=50, random_state=42)

# Predict
probs = model.predict_proba(sample_50)
preds = model.predict(sample_50)

label_map = {0: "Low Risk", 1: "Prediabetes", 2: "High Risk"}

for i, (row_idx, prob, pred) in enumerate(zip(sample_50.index, probs, preds), start=1):
    print(f"Sample {i} (row {row_idx}): {label_map[int(pred)]}, probs={np.round(prob, 3)}")

Sample 1 (row 1178): High Risk, probs=[0. 0. 1.]
Sample 2 (row 865): Prediabetes, probs=[0.319 0.681 0.   ]
Sample 3 (row 101): Low Risk, probs=[0.586 0.414 0.   ]
Sample 4 (row 439): Low Risk, probs=[1. 0. 0.]
Sample 5 (row 58): Prediabetes, probs=[0.002 0.98  0.017]
Sample 6 (row 1120): Low Risk, probs=[0.998 0.002 0.   ]
Sample 7 (row 323): Low Risk, probs=[1. 0. 0.]
Sample 8 (row 974): High Risk, probs=[0.    0.045 0.955]
Sample 9 (row 411): Low Risk, probs=[0.98 0.02 0.  ]
Sample 10 (row 855): High Risk, probs=[0. 0. 1.]
Sample 11 (row 820): Low Risk, probs=[0.984 0.016 0.   ]
Sample 12 (row 44): High Risk, probs=[0. 0. 1.]
Sample 13 (row 49): High Risk, probs=[0.    0.054 0.946]
Sample 14 (row 849): Low Risk, probs=[1. 0. 0.]
Sample 15 (row 240): Low Risk, probs=[0.998 0.002 0.   ]
Sample 16 (row 170): Low Risk, probs=[1. 0. 0.]
Sample 17 (row 523): Low Risk, probs=[1. 0. 0.]
Sample 18 (row 765): Low Risk, probs=[0.667 0.333 0.   ]
Sample 19 (row 838): High Risk, probs=[0. 0. 1.]